# Analysis for scaffold hopping

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from plotly import express as px
from tqdm import tqdm


In [ ]:
from tools import (
    compute_uniqueness,
    compute_novelty,
    compute_unique_novelty,
)

## Load data

In [ ]:
pred_dir = Path("predictions/conditional_mol/")
files = sorted(f for f in pred_dir.rglob("*.csv") if not f.stem.endswith("conditional"))
dfs = []
for file in files:
    print(file)
    df = pd.read_csv(file, low_memory=False).reset_index(drop=True)
    # Evaluation script does not break molecules correctly, introducting these rows
    df = df[~df["fail"].fillna(0).astype(bool)].reset_index(drop=True)
    df = df.drop(columns=["index", "fail"], errors="ignore")

    df_cond = pd.read_csv(
        Path(file).parent / (Path(file).stem + "_conditional.csv"), low_memory=False
    ).reset_index(drop=True)
    df_cond.columns = [c.lower().replace(" ", "_") for c in df_cond.columns]
    df_cond = df_cond.drop(columns=["index", "fail", "error"], errors="ignore")

    if len(df) != len(df_cond):
        print(f"Lengths different: {len(df)} != {len(df_cond)}")
        continue
    df = pd.concat([df, df_cond], axis=1, ignore_index=False)

    attempt = int(Path(file).parent.stem.split("_")[1])
    df["table"] = "time" if attempt == 1 else "var"
    df["method"] = Path(file).stem
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

df["total_number"] = True
df["valid"] = df["connected"] & df["chemical"] & df["physical"]

In [ ]:
# load references
truth = pd.read_csv(
    "/homes/buttensc/Projects/semla-flow/data/conditional_mol/test_first_1000.csv"
)
truth = truth[truth.fail != 1.0]
reference_smiles = set(truth["smiles"].values)

df_train = pd.read_csv("data/unconditional/geom-drugs/train.csv")
reference_smiles = set(df_train["smiles"].values)

print(len(reference_smiles))

In [ ]:
# enrichment
df["total_number"] = True
df["valid"] = df["connected"] & df["chemical"] & df["physical"]
df["novel"] = df["smiles"].map(lambda x: x not in reference_smiles)
df["valid_novel"] = df["valid"] & df["novel"]
df["valid_smiles"] = df["valid"].astype(bool) * df["smiles"]
df["valid_scaffold_rdkit_csk"] = df["valid"].astype(bool) & df[
    "scaffold_rdkit_csk"
].astype(bool)
df["valid_scaffold_hop_smiles"] = (~df["valid_scaffold_rdkit_csk"]) * df["valid_smiles"]

In [ ]:
df["integration steps"] = df["method"].str.split("_").str[1].str[1:].astype(int)
df["sigma"] = df["method"].str.split("_").str[2].str[1:].astype(str)

In [ ]:
aggs = {
    "tanimoto": ("tanimoto", "max"),
    "sucos": ("sucos", "max"),
    "scaffold_conserved": ("scaffold_rdkit_csk", "max"),
}
df_best = (
    df[df.valid & df.novel]
    .groupby(
        [
            "table",
            "method",
            "integration steps",
            "sigma",
            "reference_molecule",
            "smiles",
        ],
        observed=True,
    )
    .agg(**aggs)
    .reset_index()
)
# df_best.to_csv("predictions/conditional_mol/unique_novel_valid.csv", index=False)
df_best

## Tables

In [ ]:
df.columns

### Validity

In [ ]:
def mean(x):
    n = 100000
    if len(x) == 0:
        return float("nan")
    if len(x) > n:
        return np.mean(x)
    return np.sum(x) / n


def std(x):
    n = 100000
    if len(x) == 0:
        return float("nan")
    if len(x) > n:
        return np.std(x)
    m = np.sum(x) / n
    return np.sqrt(np.sum((x - m) ** 2) / n)


aggs = {
    "total_number": ("total_number", "sum"),
    "generated": ("total_number", mean),
    "connected": ("connected", mean),
    "chemical": ("chemical", mean),
    "physical": ("physical", mean),
    "valid": ("valid", mean),
    # "valid_scaffold_hop": ("scaffold_rdkit_csk", 1 - mean),
}
df_agg = df.groupby(["table", "integration steps", "sigma"], observed=False).agg(**aggs)
cols = df_agg.columns
df_agg.style.format("{:.0f}", subset=cols[:1]).format("{:.1%}", subset=cols[1:])

In [ ]:
df_filter = df[df.valid]
aggs = {
    "qed": ["mean", "std"],
    "sa": ["mean", "std"],
    "energy_ratio": ["mean", "std"],
    "weight": ["mean", "std"],
    "num_heavy": ["mean", "std"],
    "num_rings": ["mean", "std"],
    "lipinski": ["mean", "std"],
    "logp": ["mean", "std"],
    "spacial": ["mean", "std"],
}
df_agg = df_filter.groupby(["table", "integration steps", "sigma"], observed=False).agg(
    aggs
)
cols = df_agg.columns
df_agg.style.format("{:.2f}", subset=cols)

In [ ]:
df_filter = df[df.valid]
df_filter = df
aggs = {
    "ensemble_avg_energy": ["mean", "std"],
    "mol_pred_energy": ["mean", "std"],
    "energy_ratio": ["mean", "std"],
}
df_agg = df_filter.groupby(["table", "integration steps", "sigma"], observed=False).agg(
    aggs
)
cols = df_agg.columns
df_agg.style.format("{:.2f}", subset=cols)

In [ ]:
n = 100000

df_filter = df
aggs = {
    "Valid": ("valid", lambda x: sum(x) / n),
    # "Valid & Scaffold Hop": ("scaffold_rdkit_csk", lambda x: 1 - mean(x)),
    # "Valid & Unique": (
    #     "valid_smiles",
    #     lambda x: compute_uniqueness(x, total=1) / n,
    # ),
    # "Valid & Novel": (
    #     "valid_smiles",
    #     lambda x: compute_novelty(x, reference_smiles, total=1) / n,
    # ),
    "Valid & Unique & Novel": (
        "valid_smiles",
        lambda x: compute_unique_novelty(x, reference_smiles, total=1) / n,
    ),
    "Valid & Unique & Novel & Scaffold Hop": (
        "valid_scaffold_hop_smiles",
        lambda x: compute_unique_novelty(x, reference_smiles, total=1) / n,
    ),
}
df_agg = df_filter.groupby(["table", "integration steps", "sigma"], observed=False).agg(
    **aggs
)
cols = df_agg.columns
df_agg.style.format("{:.2%}", subset=cols)

In [ ]:
threshold_tanimoto = 0.8
threshold_sucos = 0.55

df_filter = df[df.valid]
df_filter = df
aggs = {
    "tanimoto mean": ("tanimoto", "mean"),
    "tanimoto std": ("tanimoto", "std"),
    f"tanimoto > {threshold_tanimoto}": (
        "sucos",
        lambda x: sum(x > threshold_tanimoto) / n,
    ),
    "sucos mean": ("sucos", "mean"),
    "sucos std": ("sucos", "std"),
    f"sucos > {threshold_sucos}": ("sucos", lambda x: sum(x > threshold_sucos) / n),
}
df_agg = df_filter.groupby(["table", "integration steps", "sigma"], observed=False).agg(
    **aggs
)
df_agg[f"tanimoto > {threshold_tanimoto} and sucos > {threshold_sucos}"] = (
    df_agg[f"sucos > {threshold_sucos}"] * df_agg[f"tanimoto > {threshold_tanimoto}"]
)
cols = df_agg.columns
df_agg.style.format("{:.2%}", subset=cols)

In [ ]:
threshold_sucos = 0.7
aggs = {
    "number_reference_mols": (
        "sucos",
        lambda x: sum(x > threshold_sucos) > 0,
    ),
}
print(
    f"number of reference molecules for which valid unique molecules with sucos > {threshold_sucos} were generated"
)
df_agg = df_best.groupby(
    ["table", "integration steps", "sigma", "reference_molecule"]
).agg(**aggs)
df_agg.groupby(["table", "integration steps", "sigma"]).sum()

In [ ]:
df_agg = df_best.groupby(["table", "reference_molecule"]).agg(**aggs)
df_agg.groupby(["table"]).sum()

In [ ]:
sns.scatterplot(data=df_best, x="tanimoto", y="sucos", hue="method", alpha=0.1)

# Plots

In [ ]:
metrics = {
    "sucos": "SuCOS",
    "tanimoto": "ECFP4 Bit Tanimoto",
    "ensemble_avg_energy": "Ensemble Average Energy",
    "mol_pred_energy": "Molecular Prediction Energy",
    "energy_ratio": "Energy Ratio",
    "sa": "Synthetic Accessability Score",
    "sa_normalized": "Synthetic Accessability Score (normalized)",
    "spacial": "Spacial Score",
    "qed": "QED",
    "logp": "LogP",
    "lipinski": "Lipinski Rule of 5",
    "num_heavy": "Number of Heavy Atoms",
    "weight": "Molecular Weight",
    "num_rings": "Number of Rings",
}

In [ ]:
# how many interesting new molecules have we created?

fig = sns.ecdfplot(
    df_best[df_best.table == "var"]
    .groupby(["table", "method", "smiles"])
    .agg({"sucos": "max"}),
    x="sucos",
    hue="method",
    complementary=True,
    # common_norm=False,
    stat="count",
    # element="step",
    # fill=False,
    # legend=True, palette="tab10", linewidth=1.5
)
fig.set(
    xlim=(0.3, 1),
    title="How many Unique Novel Molecules have SuCOS larger than x?",
    xlabel="SuCOS",
)

In [ ]:
# how many interesting new molecules have we created?

fig = sns.ecdfplot(
    # df[df.valid_novel][df.table == "table 1"][["method", metric]]
    df_best[(df_best.table == "var") & (~df_best.scaffold_conserved)]
    .groupby(["table", "method", "smiles"])
    .agg({"sucos": "max"}),
    x="sucos",
    hue="method",
    # bins=100,
    complementary=True,
    # common_norm=False,
    stat="count",
    # element="step",
    # kde=True,
    # fill=False,
    # legend=True, palette="tab10", linewidth=1.5
)
fig.set(
    xlim=(0.3, 1),
    title="How many Unique Novel Molecules have a new scaffold and SuCOS larger than x?",
    xlabel="SuCOS",
)

In [ ]:
# how many interesting new molecules have we created?

fig = sns.histplot(
    df_best[(df_best.table == "var")]
    .groupby(["table", "method", "smiles"])
    .agg({"sucos": "max"}),
    x="sucos",
    hue="method",
    # complementary=True,
    # common_norm=False,
    stat="count",
    element="step",
    fill=False,
    # legend=True, palette="tab10", linewidth=1.5
)
fig.set(
    xlim=(0, 1),
    ylim=(0, 1400),
    title="Distribution of SuCOS of Unique Novel Valid Molecules",
    xlabel="SuCOS",
)

In [ ]:
# how many interesting new molecules have we created?

fig = sns.histplot(
    df_best[(df_best.table == "var") & (~df_best.scaffold_conserved)]
    .groupby(["table", "method", "smiles"])
    .agg({"sucos": "max"}),
    x="sucos",
    hue="method",
    # complementary=True,
    # common_norm=False,
    stat="count",
    element="step",
    fill=False,
    # legend=True, palette="tab10", linewidth=1.5
)
fig.set(
    xlim=(0, 1),
    ylim=(0, 1400),
    title="Distribution of SuCOS of Unique Novel Valid Scaffold Hops",
    xlabel="SuCOS",
)

# Appendix

## Old tables

In [ ]:
cols = [
    "table",
    "method",
    # "scaffold_true_csk",
    "scaffold_rdkit_csk",
]
df_scaff = (
    df[df.valid_novel][cols].groupby(["table", "integration steps", "sigma"]).mean()
)
df_scaff.style.format("{:.2%}")

# Old plots

In [ ]:
metric = "sucos"
# metric = "tanimoto"
name = metrics[metric]
sns.histplot(
    df[df.valid_novel][df.table == "table 1"][["method", metric]].reset_index(
        drop=True
    ),
    x=metric,
    hue="method",
    bins=100,
    # cumulative=True,
    common_norm=False,
    stat="density",
    element="step",
    # kde=True,
    fill=False,
    # legend=True, palette="tab10", linewidth=1.5
)

In [ ]:
metric = "sucos"
# metric = "tanimoto"
name = metrics[metric]
sns.histplot(
    df[(df.valid_novel & (df.table == "table 2"))][["method", metric]].reset_index(
        drop=True
    ),
    x=metric,
    hue="method",
    bins=100,
    # cumulative=True,
    common_norm=False,
    stat="density",
    element="step",
    # kde=True,
    fill=False,
    # legend=True, palette="tab10", linewidth=1.5
)

In [ ]:
# metric = "sucos"
metric = "tanimoto"
name = metrics[metric]
sns.histplot(
    df[(df.valid_novel & (df.table == "table 1"))][["method", metric]].reset_index(
        drop=True
    ),
    x=metric,
    hue="method",
    bins=100,
    cumulative=True,
    common_norm=False,
    stat="density",
    element="step",
    # kde=True,
    fill=False,
    # legend=True, palette="tab10", linewidth=1.5
)

In [ ]:
# metric = "sucos"
metric = "tanimoto"
name = metrics[metric]
sns.histplot(
    df[(df.valid_novel & (df.table == "table 2"))][["method", metric]].reset_index(
        drop=True
    ),
    x=metric,
    hue="method",
    bins=100,
    cumulative=True,
    common_norm=False,
    stat="density",
    element="step",
    # kde=True,
    fill=False,
    # legend=True, palette="tab10", linewidth=1.5
)

## Old code 

In [ ]:
# How much repetition is there? How unique are the generated molecules?
s = df.groupby(["table", "integration steps", "sigma"])["smiles_pred"].agg(
    compute_uniqueness
)
s.name = "Uniqueness"
s

In [ ]:
# How many of the valid generated molecules are not in the test set?
s = df.groupby(["table", "integration steps", "sigma"])["smiles_pred"].agg(
    compute_novelty
)
s.name = "Novelty"
s

In [ ]:
# How many of the valid generated molecules are in the test set?
s = (
    df.groupby(["table", "integration steps", "sigma"])["smiles_pred"].agg(
        compute_novelty
    )
    - 1
).abs()
s.name = "In Test Set"
s

In [ ]:
# How many valid, unique and new molecules have we generated?
s = df.groupby(["table", "integration steps", "sigma"])["smiles_pred"].agg(
    compute_unique_novelty
)
s.name = "Unique Novelty"
s